#### Run common functions notebook

In [0]:
%run ../utils/common_functions

#### Create Dimension Tables

##### Customer Table

In [0]:
# Fetch customers data and geo location data from silver schema
customers_df = read_table("customers", "silver").dropDuplicates(["customer_id"])
geolocation_df = read_table("geolocation_details", "silver")

# Join both tables using the zip_code_prefix as key
dim_customers_df = customers_df. \
                    join(broadcast(geolocation_df), customers_df.customer_zip_code_prefix == geolocation_df.geolocation_zip_code_prefix, how = "left") \
                    .select("customer_id", "customer_unique_id", "state", "customer_city", "customer_zip_code_prefix", "latitude", "longitude") 

# Transform the dataframe
customer_rename_map = {
    "state": "customer_state",
    "latitude": "customer_lat",
    "longitude": "customer_long"
}

customer_caps_list = ["customer_city", "customer_state"]

dim_customers_df = clean_columns(dim_customers_df, rename_map = customer_rename_map, init_map = customer_caps_list)

# Write table into gold schema
write_table(df = dim_customers_df, table_name = "dim_customer", schema_name = "gold", zorder_by = "customer_state")

##### Product Table

In [0]:
# Fetch the product data from silver schema
product_df = read_table("products", "silver").dropDuplicates(["product_id"])

# Transform the dataframe
product_rename_map = {
    "category_name_eng": "product_category"
}

product_caps_list = ["product_category"]

product_drops = [
    "product_description_lenght", 
    "product_name_lenght", 
    "_ingested_at", 
    "_source_file"
]

dim_product_df = clean_columns(product_df, rename_map = product_rename_map, init_map = product_caps_list, drop_cols = product_drops)

# Wrie table into gold schema
write_table(df = dim_product_df, table_name = "dim_product", schema_name = "gold", zorder_by = "product_category")

##### Sellers Table

In [0]:
# Fetch sellers table from silver schema
sellers_df = read_table("sellers", "silver").dropDuplicates(["seller_id"])

# Join Sellers and Geo Location to get Latitude and Longitude
dim_sellers_df = sellers_df \
                    .join(broadcast(geolocation_df), sellers_df.seller_zip_code_prefix == geolocation_df.geolocation_zip_code_prefix, how = "left") \
                    .select("seller_id", "seller_zip_code_prefix", "state", "seller_city", "latitude", "longitude")

# Transform the dataframe
sellers_rename_map = {
    "state": "seller_state",
    "latitude": "seller_lat",
    "longitude": "seller_long"
}

sellers_init_map = ["seller_city", "seller_state"]

dim_sellers_df = clean_columns(dim_sellers_df, rename_map=sellers_rename_map, init_map=sellers_init_map)

# Write table into gold schema
write_table(df = dim_sellers_df, table_name = "dim_seller", schema_name = "gold", zorder_by = "seller_state")

#### Create Fact Tables

##### Sales Table

In [0]:
# Fetch the orders, order items and order payments data from silver schema
orders_df = read_table("orders", "silver").dropDuplicates(["order_id"])
order_items_df = read_table("order_items", "silver")
order_payments_df = read_table("order_payments", "silver")

In [0]:
# Aggregate Order Items Table
agg_order_items_df = order_items_df \
                        .groupBy("order_id", "product_id", "seller_id") \
                        .agg(
                            sum("price").alias("item_revenue"),
                            sum("freight_value").alias("item_freight"),
                            sum("total_value").alias("item_total_value"),
                            count("order_item_id").alias("item_quantity"))

# Aggregate Order Payments Table
agg_order_payments_df = order_payments_df \
                            .groupBy("order_id") \
                            .agg(
                                round(sum("payment_value"),2).alias("total_order_payment_value"),
                                countDistinct("payment_type").alias("total_unique_payment_types"),
                                collect_set("payment_type").alias("payment_types_list"),
                                max("payment_installments").alias("max_payment_installments"))

# Join Orders, Order Items and Order Payments to get the final fact table
fact_sales_df = orders_df \
    .join(agg_order_items_df, on = "order_id", how = "inner") \
    .join(agg_order_payments_df, on = "order_id", how = "left")

# Select only required columns from fact_sales table
fact_sales_df = fact_sales_df \
                    .select(
                        "order_id",
                        "customer_id",
                        "order_status",
                        "product_id",
                        "seller_id",
                        "item_revenue",
                        "item_freight",
                        "item_total_value",
                        "item_quantity",
                        "total_order_payment_value",
                        "total_unique_payment_types",
                        "payment_types_list",
                        "max_payment_installments")

# Write table into gold schema
write_table(df = fact_sales_df, table_name = "fact_sales", schema_name = "gold")

##### Reviews Table

In [0]:
# Fetch the order reviews data from silver schema
order_reviews_df = read_table("order_reviews", "silver").dropDuplicates(["review_id"])

# Join orders and order reviews data
fact_reviews_df = order_reviews_df \
                    .join(orders_df, on= "order_id", how = "inner") \
                    .select("review_id", "order_id", "customer_id", "order_status","review_score", "review_comment_title", "review_comment_message", "review_creation_date", "review_answer_timestamp", "has_review_commnents")

# Write table into gold schema
write_table(df = fact_reviews_df, table_name = "fact_reviews", schema_name = "gold", zorder_by = "review_score")